# 04 — Optuna Hyperparameter Search (Tier 5)

**Search space (Naive setup):**
- `learning_rate`: log-uniform [1e-5, 1e-3]
- `dropout`: uniform [0.2, 0.6]
- `dense1_units`: categorical [512, 1024, 2048]
- `dense2_units`: categorical [128, 256, 512]
- `rotation_max`: categorical [10, 20, 30]
- `zoom_range`: uniform [0.0, 0.2]
- `focal_gamma`: uniform [1.0, 3.0]

**Compute:** ~25 trials × 12 epochs ≈ **5h pe Tier 5**.

**Output:** `experiments/resnet152v2/tier5/C_optuna/`


## 1. Setup


In [1]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted


In [2]:
# Install Optuna
!pip install optuna -q
!pip install kaleido==0.2.1


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 33.2 MB/s eta 0:00:00


In [3]:
import os
import json
import time
import random
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import backend as K
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    precision_recall_fscore_support,
)

# ─── REPRODUCIBILITY ───
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print(f'TF: {tf.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')
print(f'Seed: {SEED}')

TF: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Seed: 42


In [4]:
# ─── PATHS ───
DRIVE_BASE = Path('/content/drive/MyDrive/Thesis')
PIPELINE_DIR = DRIVE_BASE / 'pipeline_outputs'

TFRECORDS_DIR = PIPELINE_DIR / 'step4_tfrecords'   # val/, test/, tier5/
STEP2_DIR = PIPELINE_DIR / 'step2'                 # tier{N}_train.csv
EXPERIMENTS_DIR = DRIVE_BASE / 'experiments' / 'resnet152v2'
EXPERIMENTS_DIR.mkdir(parents=True, exist_ok=True)

# ─── HYPERPARAMETERS ───
IMG_SIZE = 224
NUM_CLASSES = 7
BATCH_SIZE = 32
DENSE1_UNITS = 1024
DENSE2_UNITS = 512
DROPOUT_RATE = 0.3
ROTATION_MAX_DEG = 20
ZOOM_RANGE = 0.1

PAPER_TRAINABLE_PARAMS = 60_817_543
PAPER_ACCURACY = 91.0

# ─── LABELS ───
IDX_TO_LABEL = {0: 'akiec', 1: 'bcc', 2: 'bkl', 3: 'df', 4: 'mel', 5: 'nv', 6: 'vasc'}
CLASS_NAMES = [IDX_TO_LABEL[i] for i in range(NUM_CLASSES)]

print(f'TFRecords:   {TFRECORDS_DIR.exists()} {TFRECORDS_DIR}')
print(f'Step2:       {STEP2_DIR.exists()} {STEP2_DIR}')
print(f'Experiments: {EXPERIMENTS_DIR}')

for split in ['val', 'test', 'tier5']:
    d = TFRECORDS_DIR / split
    if d.exists():
        shards = sorted(d.glob('*.tfrecord'))
        print(f'  {split}: {len(shards)} shards')
    else:
        print(f'  [WARNING] {split}/ missing!')

TFRecords:   True /content/drive/MyDrive/Thesis/pipeline_outputs/step4_tfrecords
Step2:       True /content/drive/MyDrive/Thesis/pipeline_outputs/step2
Experiments: /content/drive/MyDrive/Thesis/experiments/resnet152v2
  val: 3 shards
  test: 3 shards
  tier5: 21 shards


## 2. Data pipeline


In [5]:
FEATURE_DESC = {
    'image_id':    tf.io.FixedLenFeature([], tf.string),
    'label':       tf.io.FixedLenFeature([], tf.int64),
    'lesion_id':   tf.io.FixedLenFeature([], tf.string),
    'source':      tf.io.FixedLenFeature([], tf.string),
    'image_bytes': tf.io.FixedLenFeature([], tf.string),
}


def parse_example(example_proto):
    return tf.io.parse_single_example(example_proto, FEATURE_DESC)


def decode_image(parsed):
    img = tf.io.decode_jpeg(parsed['image_bytes'], channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32) / 255.0
    return img, parsed['label'], parsed['lesion_id']


def decode_with_source(parsed):
    img = tf.io.decode_jpeg(parsed['image_bytes'], channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32) / 255.0
    return img, parsed['label'], parsed['source']


def get_split_files(split_name):
    pattern = str(TFRECORDS_DIR / split_name / '*.tfrecord')
    files = sorted(tf.io.gfile.glob(pattern))
    if not files:
        raise FileNotFoundError(f'No TFRecords found for split: {split_name}')
    return files


TRAIN_FILES = get_split_files('tier5')
VAL_FILES = get_split_files('val')
TEST_FILES = get_split_files('test')

print(f'Train pool: {len(TRAIN_FILES)} shards')
print(f'Val:        {len(VAL_FILES)} shards')
print(f'Test:       {len(TEST_FILES)} shards')

Train pool: 21 shards
Val:        3 shards
Test:       3 shards


In [6]:
def get_tier_lesion_ids(tier):
    if tier == 5:
        return None
    csv_path = STEP2_DIR / f'tier{tier}_train.csv'
    df = pd.read_csv(csv_path)
    return set(df['lesion_id'].astype(str).tolist())


def make_lesion_filter(allowed_ids):
    keys = tf.constant(sorted(list(allowed_ids)), dtype=tf.string)
    values = tf.ones(len(allowed_ids), dtype=tf.int32)
    return tf.lookup.StaticHashTable(
        tf.lookup.KeyValueTensorInitializer(keys, values),
        default_value=0,
    )


for t in [1, 2, 3, 4, 5]:
    ids = get_tier_lesion_ids(t)
    if ids is None:
        print(f'Tier {t}: no filter (full tier5)')
    else:
        print(f'Tier {t}: {len(ids):,} lesion_ids')

Tier 1: 5,987 lesion_ids
Tier 2: 7,276 lesion_ids
Tier 3: 8,565 lesion_ids
Tier 4: 9,855 lesion_ids
Tier 5: no filter (full tier5)


In [7]:
# ─── Augmentation pure tf.image + scipy (fără SeedGenerator → fără bug CUDA) ───
# Bug istoric: tf.keras.layers.RandomFlip/RandomRotation cauzează
# CUDA_ERROR_INVALID_HANDLE când SeedGenerator se inițializează înainte de GPU.
# Soluția: folosim tf.image (in-graph, GPU) + scipy via tf.numpy_function (CPU).
from scipy.ndimage import rotate as scipy_rotate

def _random_rotate_np(image_np, max_deg):
    """Rotație random pe CPU cu scipy."""
    import numpy as np
    angle = np.random.uniform(-max_deg, max_deg)
    return scipy_rotate(
        image_np, angle, reshape=False, order=1, mode='reflect',
    ).astype(np.float32)


def augment_image(image, rotation_deg=ROTATION_MAX_DEG, zoom=ZOOM_RANGE):
    """Augmentation: flip + rotation ±rotation_deg° + zoom ±zoom (per Nawaz spec)."""
    # Flip horizontal + vertical
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)

    # Rotation via scipy (CPU side, robust)
    image = tf.numpy_function(
        func=lambda x: _random_rotate_np(x, rotation_deg),
        inp=[image],
        Tout=tf.float32,
    )
    image.set_shape([None, None, 3])

    # Zoom prin resize + center crop
    zoom_factor = tf.random.uniform([], 1.0 - zoom, 1.0 + zoom)
    new_size = tf.cast(tf.cast(IMG_SIZE, tf.float32) / zoom_factor, tf.int32)
    image = tf.image.resize(image, [new_size, new_size])
    image = tf.image.resize_with_crop_or_pad(image, IMG_SIZE, IMG_SIZE)

    image = tf.clip_by_value(image, 0.0, 1.0)
    return image


def preprocess_for_model(img_01):
    """Input [0,1] (after /255). Apply per-model preprocessing.

    ResNet152V2 folosește tf.keras.applications.resnet_v2.preprocess_input
    care așteaptă input în range [0, 255] și returnează range specific arhitecturii.
    """
    img_255 = img_01 * 255.0
    return tf.keras.applications.resnet_v2.preprocess_input(img_255)


print('Augmentation + preprocessing definite')

Augmentation + preprocessing definite


In [8]:
def build_train_ds(tier, rotation_deg=ROTATION_MAX_DEG, zoom=ZOOM_RANGE):
    """Train ds — repeat + filter pe lesion_id pentru tier1-4."""
    allowed_ids = get_tier_lesion_ids(tier)

    # Augmentation layer per call (allows different rotation/zoom in Optuna)
    # NOTE: augmentation customized per call ignored — folosim augment_image global
    # cu rotation_deg și zoom din parametri
    aug = lambda img, training=True: augment_image(
        img, rotation_deg=rotation_deg, zoom=zoom,
    )
    ds = tf.data.TFRecordDataset(TRAIN_FILES, num_parallel_reads=tf.data.AUTOTUNE)
    ds = ds.map(parse_example, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(decode_image, num_parallel_calls=tf.data.AUTOTUNE)

    if allowed_ids is not None:
        table = make_lesion_filter(allowed_ids)
        ds = ds.filter(lambda img, lbl, lid: tf.equal(table.lookup(lid), 1))

    ds = ds.map(lambda img, lbl, lid: (img, lbl), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.repeat()
    ds = ds.shuffle(buffer_size=2000, seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(
        lambda img, lbl: (aug(img, training=True), lbl),
        num_parallel_calls=tf.data.AUTOTUNE,
    )
    ds = ds.map(
        lambda img, lbl: (preprocess_for_model(img), lbl),
        num_parallel_calls=tf.data.AUTOTUNE,
    )
    ds = ds.batch(BATCH_SIZE, drop_remainder=True)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds


def build_eval_ds(files, with_source=False):
    ds = tf.data.TFRecordDataset(files, num_parallel_reads=tf.data.AUTOTUNE)
    ds = ds.map(parse_example, num_parallel_calls=tf.data.AUTOTUNE)
    if with_source:
        ds = ds.map(decode_with_source, num_parallel_calls=tf.data.AUTOTUNE)
        ds = ds.map(
            lambda img, lbl, src: (preprocess_for_model(img), lbl, src),
            num_parallel_calls=tf.data.AUTOTUNE,
        )
    else:
        ds = ds.map(decode_image, num_parallel_calls=tf.data.AUTOTUNE)
        ds = ds.map(lambda img, lbl, lid: (img, lbl), num_parallel_calls=tf.data.AUTOTUNE)
        ds = ds.map(
            lambda img, lbl: (preprocess_for_model(img), lbl),
            num_parallel_calls=tf.data.AUTOTUNE,
        )
    ds = ds.batch(BATCH_SIZE, drop_remainder=False)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds


print('Dataset builders definite')

Dataset builders definite


In [9]:
def count_tier_samples(tier):
    allowed_ids = get_tier_lesion_ids(tier)
    ds = tf.data.TFRecordDataset(TRAIN_FILES, num_parallel_reads=tf.data.AUTOTUNE)
    ds = ds.map(parse_example, num_parallel_calls=tf.data.AUTOTUNE)
    if allowed_ids is not None:
        table = make_lesion_filter(allowed_ids)
        ds = ds.filter(lambda x: tf.equal(table.lookup(x['lesion_id']), 1))
    counter = Counter()
    for ex in ds:
        counter[int(ex['label'].numpy())] += 1
    return dict(counter)


TIER_DISTRIBUTIONS = {}
print('Counting samples per tier (~1-2 min):')
for t in [1, 2, 3, 4, 5]:
    dist = count_tier_samples(t)
    TIER_DISTRIBUTIONS[t] = dist
    total = sum(dist.values())
    print(f'  Tier {t}: {total:>6,} samples')

val_count = sum(1 for _ in tf.data.TFRecordDataset(VAL_FILES))
test_count = sum(1 for _ in tf.data.TFRecordDataset(TEST_FILES))
print(f'\nVal:  {val_count:,}')
print(f'Test: {test_count:,}')

Counting samples per tier (~1-2 min):
  Tier 1:  8,042 samples
  Tier 2: 10,996 samples
  Tier 3: 14,061 samples
  Tier 4: 17,300 samples
  Tier 5: 20,299 samples

Val:  2,519
Test: 2,513


## 3. Focal loss + Model


In [10]:
def compute_alpha_inverse_frequency(distribution, normalize=True):
    """sklearn 'balanced' alpha. Output: array de shape (num_classes,)."""
    counts = np.array([distribution[i] for i in range(NUM_CLASSES)], dtype=np.float32)
    total = counts.sum()
    alpha = total / (NUM_CLASSES * counts)
    if normalize:
        alpha = alpha / alpha.max()
    return alpha


def categorical_focal_loss(alpha, gamma=2.0):
    """Categorical focal loss cu alpha per clasa.

    FL(p_t) = -α_t · (1 - p_t)^γ · log(p_t)

    Folosim sparse labels (int) → one-hot intern.
    """
    alpha = tf.constant(alpha, dtype=tf.float32)

    def loss(y_true, y_pred):
        # y_true vine ca int64 (labels), y_pred ca probabilități softmax
        y_true_oh = tf.one_hot(tf.cast(y_true, tf.int32), depth=NUM_CLASSES)
        y_pred = tf.clip_by_value(y_pred, K.epsilon(), 1.0 - K.epsilon())
        ce = -y_true_oh * tf.math.log(y_pred)
        focal_weight = tf.pow(1.0 - y_pred, gamma)
        weighted_loss = alpha * focal_weight * ce
        return tf.reduce_mean(tf.reduce_sum(weighted_loss, axis=-1))

    return loss


# Sanity check pe Tier 5
alpha_t5 = compute_alpha_inverse_frequency(TIER_DISTRIBUTIONS[5])
print('Alpha for Tier 5 (normalized to max=1):')
for i, name in enumerate(CLASS_NAMES):
    print(f'  {name}: {alpha_t5[i]:.4f}')

Alpha for Tier 5 (normalized to max=1):
  akiec: 0.1544
  bcc: 0.0691
  bkl: 0.0865
  df: 1.0000
  mel: 0.0507
  nv: 0.0179
  vasc: 0.8804


In [11]:
def build_resnet152v2(
    dense1=DENSE1_UNITS, dense2=DENSE2_UNITS, dropout=DROPOUT_RATE,
    base_trainable=True, verbose=True,
):
    """Strict Nawaz Table 3 — Total 60,964,359 / Trainable 60,817,543 (paper)."""
    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

    base = tf.keras.applications.ResNet152V2(
        include_top=False,
        weights='imagenet',
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
    )
    base.trainable = base_trainable

    # CRITICAL: BN-urile din bază înghețate COMPLET (gamma+beta non-trainable).
    for layer in base.layers:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False

    x = base(inputs, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(dense1, activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(dropout)(x)
    x = tf.keras.layers.Dense(dense2, activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(dropout)(x)
    outputs = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)

    model = tf.keras.Model(inputs, outputs, name='ResNet152V2')
    model.base_model = base

    if verbose:
        total = model.count_params()
        trainable = sum(tf.size(w).numpy() for w in model.trainable_weights)
        print(f'  Total: {total:,} | Trainable: {trainable:,}')
    return model


def set_phase_1(model):
    """Phase 1: head only."""
    model.base_model.trainable = False
    trainable = sum(tf.size(w).numpy() for w in model.trainable_weights)
    print(f'[Phase 1] Trainable: {trainable:,}')


def set_phase_2(model):
    """Phase 2: full unfreeze with BN STILL frozen."""
    model.base_model.trainable = True
    for layer in model.base_model.layers:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False
    trainable = sum(tf.size(w).numpy() for w in model.trainable_weights)
    print(f'[Phase 2] Trainable: {trainable:,} (BN-uri din bază înghețate)')


_test = build_resnet152v2()
del _test
tf.keras.backend.clear_session()

  Total: 60,964,359 | Trainable: 60,673,799


## 4. Optuna search (Tier 5)


In [12]:
TIER = 5
N_TRIALS = 25
TIMEOUT_SECONDS = 36000   # 10h hard cap
EPOCHS_PER_TRIAL = 12     # max — pruner taie ne-promițătoarele
PATIENCE = 4

OPTUNA_DIR = EXPERIMENTS_DIR / f'tier{TIER}' / 'C_optuna'
OPTUNA_DIR.mkdir(parents=True, exist_ok=True)
STUDY_DB = OPTUNA_DIR / 'study.db'
print(f'Tier: {TIER}')
print(f'Trials target: {N_TRIALS}')
print(f'Output: {OPTUNA_DIR}')


Tier: 5
Trials target: 25
Output: /content/drive/MyDrive/Thesis/experiments/resnet152v2/tier5/C_optuna


In [13]:
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

class OptunaPruningCallback(tf.keras.callbacks.Callback):
    """Reportează val_accuracy către Optuna și permite pruning."""
    def __init__(self, trial, monitor='val_accuracy'):
        super().__init__()
        self.trial = trial
        self.monitor = monitor

    def on_epoch_end(self, epoch, logs=None):
        score = logs.get(self.monitor)
        if score is not None:
            self.trial.report(score, epoch)
            if self.trial.should_prune():
                self.model.stop_training = True
                raise optuna.TrialPruned()


def objective(trial):
    # ─── Search space (Naive setup) ───
    lr = trial.suggest_float('learning_rate', 1e-6, 1e-4, log=True)
    dropout = trial.suggest_float('dropout', 0.2, 0.6)
    dense1 = trial.suggest_categorical('dense1_units', [512, 1024, 2048])
    dense2 = trial.suggest_categorical('dense2_units', [128, 256, 512])
    rotation = trial.suggest_categorical('rotation_max', [10, 20, 30])
    zoom = trial.suggest_float('zoom_range', 0.0, 0.2)
    focal_gamma = trial.suggest_float('focal_gamma', 1.0, 3.0)

    # ─── Datasets cu augmentation params din trial ───
    train_ds = build_train_ds(TIER, rotation_deg=rotation, zoom=zoom)
    val_ds = build_eval_ds(VAL_FILES)

    train_samples = sum(TIER_DISTRIBUTIONS[TIER].values())
    steps_per_epoch = max(1, (train_samples // BATCH_SIZE) - 1)

    # ─── Model ───
    tf.keras.backend.clear_session()
    model = build_resnet152v2(
        dense1=dense1, dense2=dense2, dropout=dropout, verbose=False,
    )

    alpha = compute_alpha_inverse_frequency(TIER_DISTRIBUTIONS[TIER])
    loss_fn = categorical_focal_loss(alpha=alpha, gamma=focal_gamma)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr, clipnorm=1.0),
        loss=loss_fn,
        metrics=['accuracy'],
    )

    history = model.fit(
        train_ds, validation_data=val_ds,
        epochs=EPOCHS_PER_TRIAL,
        steps_per_epoch=steps_per_epoch,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(
                monitor='val_accuracy', mode='max',
                patience=PATIENCE, restore_best_weights=True,
            ),
            OptunaPruningCallback(trial),
        ],
        verbose=0,
    )

    best_acc = max(history.history.get('val_accuracy', [0]))

    del model, train_ds, val_ds
    tf.keras.backend.clear_session()
    import gc; gc.collect()

    return best_acc


In [14]:
# Create or resume study
study = optuna.create_study(
    study_name=f'efnet_tier{TIER}',
    storage=f'sqlite:///{STUDY_DB}',
    direction='maximize',
    sampler=TPESampler(seed=SEED),
    pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=4),
    load_if_exists=True,
)

print(f'Existing trials: {len(study.trials)}')
remaining = max(0, N_TRIALS - len(study.trials))
print(f'Will run {remaining} more trials (target: {N_TRIALS})')

if remaining > 0:
    study.optimize(
        objective,
        n_trials=remaining,
        timeout=TIMEOUT_SECONDS,
        gc_after_trial=True,
    )


[I 2026-05-10 11:56:18,501] Using an existing study with name 'efnet_tier5' instead of creating a new one.


Existing trials: 25
Will run 0 more trials (target: 25)


## 5. Best params + analysis


In [15]:
print(f'\n{"="*60}')
print(f'OPTUNA RESULTS — Tier {TIER}')
print(f'{"="*60}')
print(f'Number of finished trials: {len(study.trials)}')
print(f'\nBest trial (val_accuracy = {study.best_value:.4f}):')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')

# Save best params
with open(OPTUNA_DIR / 'best_params.json', 'w') as f:
    json.dump({
        'best_value': study.best_value,
        'best_params': study.best_params,
        'n_trials': len(study.trials),
        'tier': TIER,
    }, f, indent=2)

# Save trials log
trials_df = study.trials_dataframe()
trials_df.to_csv(OPTUNA_DIR / 'trials_log.csv', index=False)
print(f'\nSaved: best_params.json, trials_log.csv')



OPTUNA RESULTS — Tier 5
Number of finished trials: 25

Best trial (val_accuracy = 0.6598):
  learning_rate: 4.84208030072958e-06
  dropout: 0.3355080544497232
  dense1_units: 2048
  dense2_units: 256
  rotation_max: 20
  zoom_range: 0.047726122491097984
  focal_gamma: 2.931426874755216

Saved: best_params.json, trials_log.csv


In [16]:
# ─── Visualizations ───
try:
    from optuna.visualization import (
        plot_optimization_history,
        plot_param_importances,
        plot_parallel_coordinate,
    )

    fig1 = plot_optimization_history(study)
    fig1.write_image(str(OPTUNA_DIR / 'opt_history.png'))
    fig1.show()

    fig2 = plot_param_importances(study)
    fig2.write_image(str(OPTUNA_DIR / 'param_importance.png'))
    fig2.show()

    fig3 = plot_parallel_coordinate(study)
    fig3.write_image(str(OPTUNA_DIR / 'parallel_coord.png'))
    fig3.show()

    print('Plots saved')
except Exception as e:
    print(f'Plotting failed: {e}')
    print('(install kaleido pentru export: !pip install kaleido)')


Plots saved


In [17]:
# Top 5 trials
print('\nTop 5 trials by val_accuracy:')
sorted_trials = sorted(
    [t for t in study.trials if t.value is not None],
    key=lambda t: t.value, reverse=True,
)[:5]

for i, t in enumerate(sorted_trials, 1):
    print(f'\n#{i}: val_acc = {t.value:.4f}')
    for k, v in t.params.items():
        print(f'   {k}: {v}')



Top 5 trials by val_accuracy:

#1: val_acc = 0.6598
   learning_rate: 4.84208030072958e-06
   dropout: 0.3355080544497232
   dense1_units: 2048
   dense2_units: 256
   rotation_max: 20
   zoom_range: 0.047726122491097984
   focal_gamma: 2.931426874755216

#2: val_acc = 0.6209
   learning_rate: 7.5911048052827045e-06
   dropout: 0.24881529393791155
   dense1_units: 2048
   dense2_units: 256
   rotation_max: 20
   zoom_range: 0.19391692555291173
   focal_gamma: 2.550265646722229

#3: val_acc = 0.6201
   learning_rate: 3.890820642688127e-06
   dropout: 0.2954205051277003
   dense1_units: 2048
   dense2_units: 512
   rotation_max: 30
   zoom_range: 0.08635218734053472
   focal_gamma: 2.170492571304342

#4: val_acc = 0.6149
   learning_rate: 2.6587543983272713e-06
   dropout: 0.2727299868828402
   dense1_units: 2048
   dense2_units: 512
   rotation_max: 30
   zoom_range: 0.0912139968434072
   focal_gamma: 2.570351922786027

#5: val_acc = 0.6078
   learning_rate: 2.5081156860452325e-06
   d